In [0]:
# =============================================================================
# NB07 — Validation framework: split serialization
# =============================================================================
# Objective:
#   Define, validate, and persist all cross-validation splits so that NB08,
#   NB09, and NB10 use IDENTICAL folds — guaranteeing reproducibility and
#   making absence of data leakage auditable.
#
#   Splits produced:
#     1. main_cv       — 5-fold GroupKFold by doi_key (leave-one-paper-out).
#                        Primary validation scheme for all model comparisons.
#     2. temporal_holdout — most recent 20% of papers (by year parsed from DOI)
#                        held out entirely. Tests generalization to future work.
#     3. y_randomization  — y labels shuffled (seed fixed); used in NB08 as
#                        sanity check: R² must collapse to ~0 under random y.
#
#   Each split is stored as a row in ml_perovskites.cv_splits with columns:
#     split_type, fold_id, source_row_id, role  (train / val / test)
#
#   After writing, this notebook verifies:
#     - No doi_key appears in both train and val within any fold (leakage check)
#     - Class balance of y_log10_1p_final across folds (stratification check)
#     - Row counts sum to total dataset size
#
# Inputs:
#   - ml_perovskites.master   (691 rows — carries doi_key, y_log10_1p_final)
#
# Outputs:
#   - ml_perovskites.cv_splits  (NEW Delta table)
#     Columns: split_type, fold_id, source_row_id, doi_key, role
#
# Key rules:
#   - Groups = doi_key; same DOI never straddles train/val
#   - source_row_id and doi_key are provenance only — never features
#   - SEED = 42
# =============================================================================

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from pyspark.sql import SparkSession

SEED        = 42
SCHEMA      = "ml_perovskites"
N_FOLDS     = 5
HOLDOUT_FRAC = 0.20   # fraction of DOIs for temporal hold-out

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

rng = np.random.default_rng(SEED)

# ---------------------------------------------------------------------------
# 1. Load master — keep only rows with valid target and doi_key
# ---------------------------------------------------------------------------
master = spark.table(f"{SCHEMA}.master").toPandas()
print(f"master shape: {master.shape}")

# Filter to rows used for modelling (use_h2 == 1 / "true", non-null target)
# use_h2 is stored as string "true"/"false" in this dataset
def is_use_h2(series):
    """Return boolean mask; handles 'true'/'false', True/False, 1/0."""
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")

df = master[
    is_use_h2(master["use_h2"]) &
    master["y_log10_1p_final"].notna() &
    master["doi_key"].notna()
].copy()

print(f"master shape: {master.shape}")
print(f"  use_h2 dtype: {master['use_h2'].dtype}, unique: {list(master['use_h2'].dropna().unique())}")
print(f"  use_h2 == true: {is_use_h2(master['use_h2']).sum()} rows")
print(f"Rows after filter: {len(df)}")
print(f"Unique DOIs (groups): {df['doi_key'].nunique()}")

# Arrays for sklearn
X_idx  = np.arange(len(df))
y      = df["y_log10_1p_final"].values
groups = df["doi_key"].values

# ---------------------------------------------------------------------------
# 2. Split 1: main_cv — 5-fold GroupKFold by DOI
# ---------------------------------------------------------------------------
gkf    = GroupKFold(n_splits=N_FOLDS)
splits = []

for fold_id, (train_idx, val_idx) in enumerate(gkf.split(X_idx, y, groups)):
    for idx in train_idx:
        splits.append({
            "split_type":    "main_cv",
            "fold_id":       fold_id,
            "source_row_id": df.iloc[idx]["source_row_id"],
            "doi_key":       df.iloc[idx]["doi_key"],
            "role":          "train",
        })
    for idx in val_idx:
        splits.append({
            "split_type":    "main_cv",
            "fold_id":       fold_id,
            "source_row_id": df.iloc[idx]["source_row_id"],
            "doi_key":       df.iloc[idx]["doi_key"],
            "role":          "val",
        })

# ---------------------------------------------------------------------------
# 3. Split 2: temporal_holdout — hold out most-recent DOIs
# ---------------------------------------------------------------------------
# Extract year from DOI where possible (format 10.XXXX/YYYY... or similar)
# Fall back to doi_key alphabetical sort as a rough proxy if year not parseable
import re as _re

def extract_year(doi_key):
    m = _re.search(r"(19|20)\d{2}", str(doi_key))
    return int(m.group()) if m else 0

doi_years = (df.groupby("doi_key")["doi_key"]
               .first()
               .reset_index(drop=True))
doi_df = df[["doi_key"]].drop_duplicates().copy()
doi_df["year"] = doi_df["doi_key"].apply(extract_year)
doi_df = doi_df.sort_values("year", ascending=True).reset_index(drop=True)

n_holdout_dois = max(1, int(len(doi_df) * HOLDOUT_FRAC))
holdout_dois   = set(doi_df.tail(n_holdout_dois)["doi_key"])
train_dois     = set(doi_df.head(len(doi_df) - n_holdout_dois)["doi_key"])

print(f"\nTemporal hold-out: {len(holdout_dois)} DOIs held out "
      f"({n_holdout_dois}/{len(doi_df)} = {n_holdout_dois/len(doi_df)*100:.1f}%)")

for _, row in df.iterrows():
    role = "test" if row["doi_key"] in holdout_dois else "train"
    splits.append({
        "split_type":    "temporal_holdout",
        "fold_id":       0,
        "source_row_id": row["source_row_id"],
        "doi_key":       row["doi_key"],
        "role":          role,
    })

# ---------------------------------------------------------------------------
# 4. Split 3: y_randomization — shuffled labels, same CV structure
# ---------------------------------------------------------------------------
y_rand = y.copy()
rng.shuffle(y_rand)

# Store the shuffled index mapping so NB08 can reconstruct it
rand_map = {df.iloc[i]["source_row_id"]: float(y_rand[i]) for i in range(len(df))}

for fold_id, (train_idx, val_idx) in enumerate(gkf.split(X_idx, y, groups)):
    for idx in train_idx:
        splits.append({
            "split_type":    "y_randomization",
            "fold_id":       fold_id,
            "source_row_id": df.iloc[idx]["source_row_id"],
            "doi_key":       df.iloc[idx]["doi_key"],
            "role":          "train",
        })
    for idx in val_idx:
        splits.append({
            "split_type":    "y_randomization",
            "fold_id":       fold_id,
            "source_row_id": df.iloc[idx]["source_row_id"],
            "doi_key":       df.iloc[idx]["doi_key"],
            "role":          "val",
        })

# ---------------------------------------------------------------------------
# 5. Build DataFrame and verify
# ---------------------------------------------------------------------------
cv_splits = pd.DataFrame(splits)
print(f"\ncv_splits shape: {cv_splits.shape}")
print(f"\nRow counts by split_type and role:")
print(cv_splits.groupby(["split_type", "role"]).size().to_string())

# Leakage check: for main_cv, no doi_key in both train and val in same fold
print("\n--- Leakage check (main_cv) ---")
leakage_found = False
main_cv = cv_splits[cv_splits["split_type"] == "main_cv"]
for fold_id in range(N_FOLDS):
    fold = main_cv[main_cv["fold_id"] == fold_id]
    train_dois_fold = set(fold[fold["role"] == "train"]["doi_key"])
    val_dois_fold   = set(fold[fold["role"] == "val"]["doi_key"])
    overlap = train_dois_fold & val_dois_fold
    if overlap:
        print(f"  LEAKAGE in fold {fold_id}: {overlap}")
        leakage_found = True
    else:
        train_n = fold[fold["role"] == "train"]["source_row_id"].nunique()
        val_n   = fold[fold["role"] == "val"]["source_row_id"].nunique()
        val_dois_n = len(val_dois_fold)
        print(f"  Fold {fold_id}: {train_n} train rows, {val_n} val rows, "
              f"{val_dois_n} val DOIs — no leakage ✓")

if not leakage_found:
    print("All folds: no DOI leakage ✓")

# Row count check
main_row_ids = set(main_cv[main_cv["fold_id"] == 0]["source_row_id"])
print(f"\nDataset size (from fold 0 coverage): {len(df)} rows")

# ---------------------------------------------------------------------------
# 6. Write cv_splits to Delta
# ---------------------------------------------------------------------------
spark_df = spark.createDataFrame(cv_splits)
(spark_df.write
 .format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{SCHEMA}.cv_splits"))

rc = spark.table(f"{SCHEMA}.cv_splits").count()
print(f"\n✓ Written: {SCHEMA}.cv_splits ({rc} rows)")

# Also write y_randomization mapping as a small lookup table
rand_df = pd.DataFrame([
    {"source_row_id": sid, "y_rand": val}
    for sid, val in rand_map.items()
])
spark.createDataFrame(rand_df).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA}.y_randomized")

print(f"✓ Written: {SCHEMA}.y_randomized ({len(rand_df)} rows)")
print("\n=== NB07 COMPLETE ===")
print("Next: NB08 — baseline models using splits from cv_splits.")